# 📂 05 - Working with Files and Paths

Data science means working with files all the time: datasets, config files, logs. In Module 04 you used `with open(...)` to loop over a file line by line. This notebook focuses on **finding** files reliably and working with common file **formats**.

## ✅ Learning Goals
- Check where you are in the filesystem, from a notebook **and** from a Python script
- Explain relative vs. absolute paths, and why "file not found" errors happen
- Build, inspect, and create paths with `pathlib`
- Choose the right file mode (`'r'`, `'w'`, `'a'`, `'x'`) and encoding
- Read and write text, JSON, and CSV files, and know the gotchas of each
- Find files with `glob`, and copy, rename, and delete them

## 1. Where Am I?

In [ ]:
# Print the current working directory
%pwd

In [ ]:
# List files in the current directory
%ls

In [ ]:
# Change to the parent directory
%cd ..
%pwd

⚠️ **Two `%cd` gotchas:**
1. **Don't put a comment on the same line as a magic.** `%cd ..  # go up` fails because IPython treats `# go up` as part of the folder name. Put comments on their own line, as above.
2. **`%cd` stays in effect** for every cell after it. Run `%cd -` to go back, or every file you write later lands in the wrong folder.

In [ ]:
%cd -
%pwd

### The Python Way
`%pwd` and `%cd` are **notebook-only** magics. They don't work in a regular `.py` script. The `pathlib` module works everywhere:

In [ ]:
from pathlib import Path

Path.cwd()

✅ **Your Turn**: Use `%cd` to go into a subfolder (any folder `%ls` shows), and list its contents. Then use `%cd -` to return, and confirm with `Path.cwd()` that you're back where you started.

In [ ]:
# Your turn here


## 2. Relative vs. Absolute Paths
- An **absolute** path starts from the root of the drive, for example `/Users/sam/Documents/data.csv` (Mac/Linux) or `C:/Users/sam/Documents/data.csv` (Windows). It always points to the same place, but it only works on **your** computer.
- A **relative** path starts from the **current working directory**: `data.csv`, `data/scores.csv`, or `../other_folder/notes.txt`. Here `..` means "the parent folder" and `.` means "this folder".

Most `FileNotFoundError`s come from a relative path that's being resolved from a different folder than you think. When in doubt, check `Path.cwd()`.

💡 Use **relative** paths in shared code and assignments. An absolute path to your own home folder breaks on everyone else's computer, including your grader's.

In [ ]:
p = Path('data') / 'scores.csv'
print(p)                # relative
print(p.resolve())      # absolute: the full path, built from the current directory

### Building and Inspecting Paths with `pathlib`
The `/` operator joins path pieces using the right separator for your operating system, so the same code works on Windows, Mac, and Linux.

In [ ]:
p = Path('data') / '2026' / 'sales_january.csv'

print('name:  ', p.name)     # file name with extension
print('stem:  ', p.stem)     # file name without extension
print('suffix:', p.suffix)   # extension
print('parent:', p.parent)   # folder that contains it
print('exists:', p.exists())

⚠️ **Windows backslash trap.** In a normal Python string, `\` starts an escape sequence, so `'C:\new\test.csv'` actually contains a **newline** (`\n`) and a **tab** (`\t`). Use forward slashes (`'C:/new/data.csv'`, which works on Windows too), a raw string (`r'C:\new\test.csv'`), or `pathlib`.

In [ ]:
print('C:\new\test.csv')     # \n is a newline, \t is a tab!
print(r'C:\new\test.csv')    # raw string: what you meant

### Creating Folders
`mkdir(exist_ok=True)` creates a folder and doesn't complain if it already exists. Add `parents=True` to create any missing folders along the way.

Every file this notebook creates goes into a **`05-OutputFiles`** folder, through the `OUT` variable below. Upload that folder along with this notebook when you submit.

In [ ]:
OUT = Path('05-OutputFiles')
OUT.mkdir(exist_ok=True)

# parents=True creates both 'notes' and 'drafts' if they don't exist yet
(OUT / 'notes' / 'drafts').mkdir(parents=True, exist_ok=True)

OUT.resolve()

✅ **Your Turn**: Build a path to `OUT / 'reports' / 'final_report.pdf'`. Print its `name`, `stem`, `suffix`, and `parent`, and check whether it `exists()`. (It shouldn't, since you only built the path. A path is just a name until you create something there.)

In [ ]:
# Your turn here


## 3. Writing and Reading Text Files

### File Modes

| Mode | Meaning | If the file exists | If it doesn't |
|---|---|---|---|
| `'r'` | Read (the default) | Reads it | ❌ `FileNotFoundError` |
| `'w'` | Write | ⚠️ **Erases it** first | Creates it |
| `'a'` | Append | Adds to the end | Creates it |
| `'x'` | Exclusive create | ❌ `FileExistsError` | Creates it |

Add `'b'` (for example `'rb'`) for **binary** files like images. Use `'x'` when overwriting a file would be a disaster.

### Encoding
Always pass `encoding='utf-8'` for text files. Mac and Linux default to UTF-8, but **Windows** often doesn't, so a file containing accents or emoji (é, ñ, 🐍) can fail to read or come out garbled there.

In [ ]:
# Writing to a text file
with open(OUT / 'example.txt', 'w', encoding='utf-8') as f:
    f.write('Hello, Jupyter!\n')
    f.write('This is a text file. 🐍\n')

In [ ]:
# Reading the whole file into one string
with open(OUT / 'example.txt', 'r', encoding='utf-8') as f:
    contents = f.read()
contents

There are several ways to read, depending on what you need:

In [ ]:
with open(OUT / 'example.txt', encoding='utf-8') as f:
    lines = f.readlines()      # list of lines, each still ending in '\n'
print(lines)

with open(OUT / 'example.txt', encoding='utf-8') as f:
    for line in f:             # one line at a time: best for big files
        print(line.strip())    # .strip() removes the trailing '\n'

### The `pathlib` Shortcut
For small files, `Path` can read or write the whole thing in one line. It opens and closes the file for you:

In [ ]:
note = OUT / 'quick_note.txt'
note.write_text('Written with pathlib!\n', encoding='utf-8')
note.read_text(encoding='utf-8')

✅ **Your Turn**: Write your name into `OUT / 'me.txt'`, then read it back and print the result. Try it once with `open()` and once with `write_text()`/`read_text()`.

In [ ]:
# Your turn here


## 4. JSON Files

In [ ]:
import json

data = {'name': 'Graylian', 'type': 'mascot', 'skills': ['howl', 'parrot talk']}

# Write JSON. indent=2 makes the file human-readable
with open(OUT / 'data.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2)

# Read JSON
with open(OUT / 'data.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)
loaded

Here's what the file actually looks like on disk:

In [ ]:
print((OUT / 'data.json').read_text(encoding='utf-8'))

### JSON Isn't Exactly Python
A round trip through JSON can quietly change your data:
- **Tuples come back as lists.** JSON only has arrays.
- **Dictionary keys become strings.** `{1: 'a'}` comes back as `{'1': 'a'}`.
- `True`/`False`/`None` are stored as `true`/`false`/`null`.
- Some types can't be saved at all, such as sets and dates. Those raise a `TypeError`.

`json.dumps()` / `json.loads()` (with an **s**, for "string") do the same conversion to and from a string instead of a file:

In [ ]:
original = {1: 'one', 'point': (3, 4), 'done': True, 'missing': None}

text = json.dumps(original)
print(text)

back = json.loads(text)
print(back)
print('Same as before?', back == original)

✅ **Your Turn**: Create a dictionary with your name and 2-3 favorite courses. Save it to `OUT / 'courses.json'` with `indent=2`, then read it back.

In [ ]:
# Your turn here


## 5. CSV Files

In [ ]:
import csv

# Writing CSV
rows = [['Name', 'Score'], ['Alice', 90], ['Bob', 85]]
with open(OUT / 'scores.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(rows)

# Reading CSV
with open(OUT / 'scores.csv', 'r', newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

💡 `newline=''` stops the `csv` module from adding blank lines between rows on Windows. Always use it with CSV files.

### Gotcha: Everything Comes Back as a String
Notice the output above: `'90'`, not `90`. CSV files are just text, so you have to convert numbers yourself before doing math:

In [ ]:
with open(OUT / 'scores.csv', newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)          # grab the header row first
    scores = [int(row[1]) for row in reader]

print(header)
print(scores, '-> average:', sum(scores) / len(scores))

### Why Not Just Use `.split(',')`?
Because real data contains commas. The `csv` module wraps such values in quotes and reads them back correctly. A plain `split` breaks them apart:

In [ ]:
with open(OUT / 'names.csv', 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows([['Name', 'City'], ['Smith, John', 'Wichita Falls']])

line = (OUT / 'names.csv').read_text(encoding='utf-8').splitlines()[1]
print('Raw line:   ', line)
print('split(","): ', line.split(','))
print('csv.reader: ', next(csv.reader([line])))

(In notebook 06, Pandas handles all of this, including the type conversion, in a single `pd.read_csv()` call.)

✅ **Your Turn**: Create `OUT / 'hobbies.csv'` with a header row and 3 rows of your own data, including at least one number column. Read it back, convert the numbers, and print their total.

In [ ]:
# Your turn here


## 6. Finding and Managing Files
### Finding Files with `glob`
Real projects often have a folder full of data files. `glob` finds every file matching a pattern, where `*` means "anything":

In [ ]:
for path in sorted(OUT.glob('*.csv')):
    print(f'{path.name:15} {path.stat().st_size:5} bytes')

- `OUT.glob('*.json')`: all JSON files in `OUT`
- `OUT.glob('sales_*.csv')`: names starting with `sales_`
- `OUT.rglob('*.txt')`: **recursive**, so it also searches every subfolder
- `OUT.iterdir()`: everything in the folder, files and folders both

### Copying, Renaming, and Deleting

In [ ]:
import shutil

temp = OUT / 'temp.txt'
temp.write_text('temporary\n', encoding='utf-8')

shutil.copy(temp, OUT / 'temp_copy.txt')     # copy
temp.rename(OUT / 'temp_renamed.txt')        # rename (or move)

print(sorted(p.name for p in OUT.glob('temp*')))

# Delete both. missing_ok=True means "no error if it's already gone"
(OUT / 'temp_copy.txt').unlink(missing_ok=True)
(OUT / 'temp_renamed.txt').unlink(missing_ok=True)
print(sorted(p.name for p in OUT.glob('temp*')))

⚠️ `unlink()` deletes **permanently**. There's no Recycle Bin or Trash. To delete a whole folder and everything in it, use `shutil.rmtree(folder)`, carefully.

✅ **Your Turn**: Use `glob` to list every `.json` file in `OUT`. Then use `rglob('*')` to list **everything** in `OUT`, including the empty `notes/drafts` folder you made earlier.

In [ ]:
# Your turn here


---
### Summary
- `%pwd`/`%cd` only work in notebooks. `Path.cwd()` works everywhere. Watch out for a `%cd` that's still in effect.
- Relative paths are resolved from the current working directory. Use them (not absolute paths) in shared code.
- Build paths with `Path` and `/`, inspect them with `.name`/`.suffix`/`.parent`, and create folders with `mkdir(parents=True, exist_ok=True)`.
- Use `with open(..., encoding='utf-8')`. `'w'` erases, `'a'` appends, `'x'` refuses to overwrite.
- JSON stores structured key/value data, but tuples come back as lists and keys come back as strings.
- CSV stores tabular data. Use the `csv` module (not `split`), use `newline=''`, and convert the strings back to numbers.
- `glob`/`rglob` find files. `shutil.copy`, `rename`, and `unlink` manage them.

## 🏋️ Practice

### Practice 1 — Append, Don't Overwrite (easy)
Open `OUT / 'example.txt'` in append mode (`'a'` instead of `'w'`) and add one more line to it without erasing what's already there. Read the whole file back to confirm. Then try opening it with mode `'x'` and explain the error you get in a comment.

In [ ]:
# Your code here


### Practice 2 — JSON Round Trip with a List (medium)
Save a list of 3 dictionaries (each with `'name'` and `'score'` keys) to `OUT / 'results.json'`, then read it back and print the average score.

In [ ]:
# Your code here


### Practice 3 — CSV to Dictionaries (harder)
Read `OUT / 'scores.csv'` back in using `csv.DictReader` instead of `csv.reader`, and print each row as `'Name got a Score'` using the column names as dictionary keys. Then use `csv.DictWriter` to write a new file, `OUT / 'scores_curved.csv'`, with every score raised by 5 points.

In [ ]:
# Your code here


## 🔥 Challenge (Optional) — Process a Whole Folder
This combines everything in this notebook with the loops from Module 04:
1. Create a folder `OUT / 'sales'` and write **three** CSV files into it, `sales_jan.csv`, `sales_feb.csv`, and `sales_mar.csv`. Each has a header `Item,Amount` and a few rows.
2. Use `glob('sales_*.csv')` to find them **without typing their names again**.
3. Read each one, add up its `Amount` column (remember: convert the strings!), and print a line like `sales_feb: $1,234.00` for each month, plus a grand total.

**Bonus:** save the monthly totals to `OUT / 'sales_summary.json'`.

In [ ]:
# Your code here


## 📦 Check Your Output Files
Run this last to see everything you created. All of it should be inside `05-OutputFiles`. Upload that folder along with this notebook.

In [ ]:
for path in sorted(OUT.rglob('*')):
    print(path)